In [ ]:
import json
import os
import torch
import random
import numpy as np
import warnings

from pathlib import Path
from typing import Any, Dict, List

from datasets import Dataset, DatasetDict, Audio
from typing import Dict, List

import pytorch_lightning as pl

import IPython.display as ipd
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor, WhisperForConditionalGeneration
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger, TensorBoardLogger
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torchaudio

import gc
from datasets import DatasetDict, Audio
from tqdm import tqdm

warnings.filterwarnings(
    "ignore",
    message="IProgress not found. Please update jupyter and ipywidgets.*",
)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Preprocessing

In [98]:
DATASET_PATH = "./dataset/"

In [99]:
test_lines = ['toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89',
'toronto_43', 'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7',
'toronto_123', 'toronto_54', 'toronto_67', 'toronto_62', 'toronto_81',
'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166',
'toronto_58']

In [100]:
def load_toronto_dataset(
    json_path: str,
    test_lines: List[str],
    val_part: float = 0.1,
    max_size: int | None = None,
) -> DatasetDict:
    if not 0.0 < val_part < 1.0:
        raise ValueError("val_part must be between 0 and 1.")

    json_path = Path(json_path)
    if not json_path.is_file():
        raise FileNotFoundError(f"JSON file not found: {json_path}")

    with json_path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    test_ids = set(test_lines)
    train_val_items: List[Dict[str, Any]] = []
    test_items: List[Dict[str, Any]] = []

    for audio_path_str, transcription in data.items():
        audio_path = Path(audio_path_str)

        if not audio_path.is_file():
            print(f"Warning: file not found - {audio_path}")
            continue

        sample_id = audio_path.stem  # e.g. "toronto_157_0"
        line_id = audio_path.parent.name or sample_id.rsplit("_", 1)[0]

        item = {
            "id": sample_id,
            "path": str(audio_path),
            "sentence": transcription,
            "audio": str(audio_path),
        }

        if line_id in test_ids:
            test_items.append(item)
        else:
            train_val_items.append(item)

    random.shuffle(train_val_items)
    random.shuffle(test_items)

    if max_size is not None:
        max_size = max(0, max_size)
        train_val_items = train_val_items[:max_size]

    split_idx = int((1.0 - val_part) * len(train_val_items))
    train_items = train_val_items[:split_idx]
    val_items = train_val_items[split_idx:]

    print(f"Train set: {len(train_items)} samples")
    print(f"Val set:   {len(val_items)} samples")
    print(f"Test set:  {len(test_items)} samples")

    dataset_dict = DatasetDict(
        {
            "train": Dataset.from_list(train_items),
            "val": Dataset.from_list(val_items),
            "test": Dataset.from_list(test_items),
        }
    )

    # Lazy decode to avoid storing duplicated audio arrays in memory.
    dataset_dict = dataset_dict.cast_column("audio", Audio(sampling_rate=16000))
    return dataset_dict

In [101]:
labels_json = DATASET_PATH + "labels.jsonl"
toronto = load_toronto_dataset(labels_json, test_lines, max_size=None)

Train set: 11484 samples
Val set:   1277 samples
Test set:  5542 samples


In [102]:
gc.collect()

5714

In [103]:
sample = toronto['train'][7]
print(f"Path: {sample['id']}")
print(f"Reference: {sample['sentence']}")
ipd.Audio(data=sample["audio"]["array"], autoplay=False, rate=sample["audio"]["sampling_rate"])

Path: toronto_161_120
Reference: аби нарешті прямо і відверто спитати – чим відрізняєсі Арктика від Антарктики?!


# Whisper

In [ ]:
whisper_token = "your_huggingface_token_here"

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-base", token=whisper_token)
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-base", language="Ukrainian", task="transcribe", token=whisper_token)
processor = WhisperProcessor.from_pretrained("openai/whisper-base", language="Ukrainian", task="transcribe", token=whisper_token)

In [ ]:
toronto_ready = toronto

gc.collect()

3

In [106]:
import importlib
import warnings

_wer_metric = None
_cer_metric = None
_jiwer_module = None

try:
    evaluate_module = importlib.import_module("evaluate")
    _wer_metric = evaluate_module.load("wer")
    _cer_metric = evaluate_module.load("cer")
except Exception:
    try:
        _jiwer_module = importlib.import_module("jiwer")
    except Exception:
        _jiwer_module = None


def _edit_distance(seq_a: List[Any], seq_b: List[Any]) -> int:
    if not seq_a:
        return len(seq_b)
    if not seq_b:
        return len(seq_a)

    prev_row = list(range(len(seq_b) + 1))
    for i, a_item in enumerate(seq_a, start=1):
        current_row = [i]
        for j, b_item in enumerate(seq_b, start=1):
            insertion = current_row[j - 1] + 1
            deletion = prev_row[j] + 1
            substitution = prev_row[j - 1] + (0 if a_item == b_item else 1)
            current_row.append(min(insertion, deletion, substitution))
        prev_row = current_row
    return prev_row[-1]


def _fallback_wer(predictions: List[str], references: List[str]) -> float:
    total_words = 0
    total_errors = 0
    for pred, ref in zip(predictions, references):
        ref_words = ref.split()
        pred_words = pred.split()
        if not ref_words:
            continue
        total_errors += _edit_distance(ref_words, pred_words)
        total_words += len(ref_words)

    if total_words == 0:
        return 0.0
    return 100.0 * total_errors / total_words


def _fallback_cer(predictions: List[str], references: List[str]) -> float:
    total_chars = 0
    total_errors = 0
    for pred, ref in zip(predictions, references):
        ref_chars = list(ref)
        pred_chars = list(pred)
        if not ref_chars:
            continue
        total_errors += _edit_distance(ref_chars, pred_chars)
        total_chars += len(ref_chars)

    if total_chars == 0:
        return 0.0
    return 100.0 * total_errors / total_chars


def compute_text_metrics(predictions: List[str], references: List[str]) -> Dict[str, float]:
    if _wer_metric is not None and _cer_metric is not None:
        return {
            "wer": 100.0 * _wer_metric.compute(predictions=predictions, references=references),
            "cer": 100.0 * _cer_metric.compute(predictions=predictions, references=references),
        }

    if _jiwer_module is not None:
        return {
            "wer": 100.0 * _jiwer_module.wer(references, predictions),
            "cer": 100.0 * _jiwer_module.cer(references, predictions),
        }

    warnings.warn(
        "Neither evaluate nor jiwer is installed; using internal WER/CER fallback.",
        RuntimeWarning,
    )
    return {
        "wer": _fallback_wer(predictions, references),
        "cer": _fallback_cer(predictions, references),
    }

In [107]:
@dataclass
class WhisperSeq2SeqCollator:
    processor: WhisperProcessor
    decoder_start_token_id: int
    max_label_length: int = 448

    def __call__(
        self,
        features: List[Dict[str, Any]],
    ) -> Dict[str, torch.Tensor]:
        audio_arrays = [f["audio"]["array"] for f in features]
        sampling_rate = features[0]["audio"]["sampling_rate"]

        input_batch = self.processor.feature_extractor(
            audio_arrays,
            sampling_rate=sampling_rate,
            return_tensors="pt",
        )

        label_batch = self.processor.tokenizer(
            [f["sentence"] for f in features],
            padding=True,
            truncation=True,
            max_length=self.max_label_length,
            return_tensors="pt",
        )

        labels = label_batch["input_ids"].masked_fill(label_batch["attention_mask"].ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().item():
            labels = labels[:, 1:]

        return {
            "input_features": input_batch["input_features"],
            "labels": labels,
        }


class LitWhisperASR(pl.LightningModule):
    def __init__(
        self,
        processor: WhisperProcessor,
        model_name: str = "openai/whisper-base",
        hf_token: str | None = None,
        lr: float = 1e-5,
        max_new_tokens: int = 225,
        freeze_encoder: bool = True,
    ) -> None:
        super().__init__()
        self.processor = processor
        self.model = WhisperForConditionalGeneration.from_pretrained(
            model_name,
            token=hf_token,
        )
        self.model.generation_config.language = "ukrainian"
        self.model.generation_config.task = "transcribe"
        self.model.generation_config.forced_decoder_ids = None

        self.model.config.use_cache = False
        self.model.gradient_checkpointing_enable()

        if freeze_encoder:
            self.model.model.encoder.requires_grad_(False)

        self.learning_rate = lr
        self.max_new_tokens = max_new_tokens

        self._val_predictions: List[str] = []
        self._val_references: List[str] = []
        self._test_predictions: List[str] = []
        self._test_references: List[str] = []

        self.save_hyperparameters(ignore=["processor"])

    def forward(self, input_features: torch.Tensor, labels: torch.Tensor | None = None):
        return self.model(input_features=input_features, labels=labels)

    def _decode_predictions_and_references(
        self,
        generated_ids: torch.Tensor,
        labels: torch.Tensor,
    ) -> tuple[List[str], List[str]]:
        labels = labels.detach().clone()
        labels[labels == -100] = self.processor.tokenizer.pad_token_id

        predictions = self.processor.batch_decode(generated_ids, skip_special_tokens=True)
        references = self.processor.batch_decode(labels, skip_special_tokens=True)

        filtered_predictions: List[str] = []
        filtered_references: List[str] = []
        for pred, ref in zip(predictions, references):
            if ref.strip():
                filtered_predictions.append(pred)
                filtered_references.append(ref)

        return filtered_predictions, filtered_references

    def training_step(self, batch: Dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        _ = batch_idx
        outputs = self(input_features=batch["input_features"], labels=batch["labels"])
        loss = outputs.loss
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def on_validation_epoch_start(self) -> None:
        self._val_predictions.clear()
        self._val_references.clear()

    def validation_step(self, batch: Dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        _ = batch_idx
        outputs = self(input_features=batch["input_features"], labels=batch["labels"])
        val_loss = outputs.loss
        self.log("val_loss", val_loss, prog_bar=True, on_step=False, on_epoch=True)

        generated_ids = self.model.generate(
            input_features=batch["input_features"],
            max_new_tokens=self.max_new_tokens,
        )

        preds, refs = self._decode_predictions_and_references(generated_ids, batch["labels"])
        self._val_predictions.extend(preds)
        self._val_references.extend(refs)

        return val_loss

    def on_validation_epoch_end(self) -> None:
        if not self._val_references:
            self.log("val_wer", 0.0, prog_bar=True)
            self.log("val_cer", 0.0, prog_bar=True)
            return

        metrics = compute_text_metrics(self._val_predictions, self._val_references)
        self.log("val_wer", metrics["wer"], prog_bar=True)
        self.log("val_cer", metrics["cer"], prog_bar=True)

    def on_test_epoch_start(self) -> None:
        self._test_predictions.clear()
        self._test_references.clear()

    def test_step(self, batch: Dict[str, torch.Tensor], batch_idx: int) -> torch.Tensor:
        _ = batch_idx
        outputs = self(input_features=batch["input_features"], labels=batch["labels"])
        test_loss = outputs.loss
        self.log("test_loss", test_loss, prog_bar=True, on_step=False, on_epoch=True)

        generated_ids = self.model.generate(
            input_features=batch["input_features"],
            max_new_tokens=self.max_new_tokens,
        )

        preds, refs = self._decode_predictions_and_references(generated_ids, batch["labels"])
        self._test_predictions.extend(preds)
        self._test_references.extend(refs)

        return test_loss

    def on_test_epoch_end(self) -> None:
        if not self._test_references:
            self.log("test_wer", 0.0, prog_bar=True)
            self.log("test_cer", 0.0, prog_bar=True)
            return

        metrics = compute_text_metrics(self._test_predictions, self._test_references)
        self.log("test_wer", metrics["wer"], prog_bar=True)
        self.log("test_cer", metrics["cer"], prog_bar=True)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.learning_rate)

In [108]:
class TorontoWhisperDataModule(pl.LightningDataModule):
    def __init__(
        self,
        train_dataset: Dataset,
        val_dataset: Dataset,
        test_dataset: Dataset,
        collator: WhisperSeq2SeqCollator,
        batch_size: int = 1,
        num_workers: int = 0,
    ) -> None:
        super().__init__()
        self.train_dataset = train_dataset
        self.val_dataset = val_dataset
        self.test_dataset = test_dataset
        self.collator = collator
        self.batch_size = batch_size
        self.num_workers = num_workers

    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            collate_fn=self.collator,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_dataset,
            batch_size=max(1, self.batch_size // 2),
            shuffle=False,
            collate_fn=self.collator,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

    def test_dataloader(self):
        return torch.utils.data.DataLoader(
            self.test_dataset,
            batch_size=max(1, self.batch_size // 2),
            shuffle=False,
            collate_fn=self.collator,
            num_workers=self.num_workers,
            pin_memory=torch.cuda.is_available(),
        )

In [109]:
import time

BATCH_SIZE = 1
NUM_WORKERS = 0
MAX_STEPS = 10000
LEARNING_RATE = 5e-6
FREEZE_ENCODER = False
RUN_TAG = time.strftime("%Y%m%d_%H%M%S") + "-enc-unfrozen-lr5e6"
OUTPUT_DIR = "./whisper-base-uk-lightning/" + RUN_TAG
CHECKPOINT_DIR = "./whisper-base-uk-checkpoints/" + RUN_TAG

collator = WhisperSeq2SeqCollator(
    processor=processor,
    decoder_start_token_id=tokenizer.convert_tokens_to_ids("<|startoftranscript|>"),
)

data_module = TorontoWhisperDataModule(
    train_dataset=toronto_ready["train"],
    val_dataset=toronto_ready["val"],
    test_dataset=toronto_ready["test"],
    collator=collator,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

lit_model = LitWhisperASR(
    processor=processor,
    model_name="openai/whisper-base",
    hf_token=whisper_token,
    lr=LEARNING_RATE,
    max_new_tokens=225,
    freeze_encoder=FREEZE_ENCODER,
)

checkpoint_callback = ModelCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename="whisper-base-uk-{epoch:02d}-{val_wer:.4f}",
    monitor="val_wer",
    mode="min",
    save_top_k=3,
    save_last=True,
)

try:
    logger = TensorBoardLogger("tb_logs", name="whisper-base-uk")
except ModuleNotFoundError:
    print("TensorBoard is not installed, using CSV logger instead.")
    logger = CSVLogger("csv_logs", name="whisper-base-uk")

val_check_interval = max(100, len(toronto_ready["train"]) // max(1, BATCH_SIZE * 4))

trainer = pl.Trainer(
    max_steps=MAX_STEPS,
    max_epochs=20,
    accelerator="gpu" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu",
    devices=1,
    precision="16-mixed" if torch.cuda.is_available() else "32-true",
    gradient_clip_val=1.0,
    accumulate_grad_batches=1,
    log_every_n_steps=25,
    val_check_interval=val_check_interval,
    callbacks=[checkpoint_callback],
    logger=logger,
)

trainer.fit(lit_model, datamodule=data_module)


Loading weights: 100%|██████████| 245/245 [00:00<00:00, 8214.59it/s]
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


TensorBoard is not installed, using CSV logger instead.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                            ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ model │ WhisperForConditionalGeneration │ 72.6 M │ eval │     0 │
└───┴───────┴─────────────────────────────────┴────────┴──────┴───────┘

Trainable params: 72.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 72.6 M                                                                                               
Total estimated model params size (MB): 290                                                                        
Modules in train mode: 0                                                                                           
Modules in eval mode: 182                                                                                          
Total FLOPs: 0

/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


/var/folders/dp/y8tgtl0x59n2jb9l03_0kjww0000gn/T/ipykernel_1090/1658798283.py:82: RuntimeWarning: Neither evaluate 
nor jiwer is installed; using internal WER/CER fallback.
  warnings.warn(

/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/co
nnectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. 
Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve 
performance.

/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/pytorch_lightning/loops/fit_
loop.py:534: Found 182 module(s) in eval mode at the start of training. This may lead to unexpected behavior during
training. If this is intentional, you can ignore this warning.

Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [110]:
if not checkpoint_callback.best_model_path:
    raise RuntimeError("No best checkpoint found. Run the training cell first.")

test_metrics = trainer.test(datamodule=data_module, ckpt_path="best")
print(test_metrics)
print(f"Best checkpoint: {checkpoint_callback.best_model_path}")

best_model = LitWhisperASR.load_from_checkpoint(
    checkpoint_callback.best_model_path,
    processor=processor,
    model_name="openai/whisper-base",
    hf_token=whisper_token,
    lr=LEARNING_RATE,
    max_new_tokens=225,
    freeze_encoder=FREEZE_ENCODER,
)
best_model.model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)


Restoring states from the checkpoint path at /Users/ihorivanyshyn/Documents/Audio_course/toronto/whisper-base-uk-checkpoints/20260422_120128-enc-unfrozen-lr5e6/whisper-base-uk-epoch=00-val_wer=48.2867.ckpt
Loaded model weights from the checkpoint at /Users/ihorivanyshyn/Documents/Audio_course/toronto/whisper-base-uk-checkpoints/20260422_120128-enc-unfrozen-lr5e6/whisper-base-uk-epoch=00-val_wer=48.2867.ckpt
/Users/ihorivanyshyn/Documents/Audio_course/toronto/.venv/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

/var/folders/dp/y8tgtl0x59n2jb9l03_0kjww0000gn/T/ipykernel_1090/1658798283.py:82: RuntimeWarning: Neither evaluate 
nor jiwer is installed; using internal WER/CER fallback.
  warnings.warn(

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_cer          │    25.187131881713867     │
│         test_loss         │    0.7623718976974487     │
│         test_wer          │     53.8280143737793      │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 0.7623718976974487, 'test_wer': 53.8280143737793, 'test_cer': 25.187131881713867}]
Best checkpoint: /Users/ihorivanyshyn/Documents/Audio_course/toronto/whisper-base-uk-checkpoints/20260422_120128-enc-unfrozen-lr5e6/whisper-base-uk-epoch=00-val_wer=48.2867.ckpt


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.42s/it]


['./whisper-base-uk-lightning/20260422_120128-enc-unfrozen-lr5e6/processor_config.json']

# Model testing



In [ ]:
test_loader = torch.utils.data.DataLoader(
    toronto_ready["test"],
    batch_size=1,
    collate_fn=collator,
    num_workers=0,
    shuffle=False,
)


def evaluate_model(
    model: WhisperForConditionalGeneration,
    processor: WhisperProcessor,
    dataloader: torch.utils.data.DataLoader,
    max_new_tokens: int = 225,
) -> Dict[str, float]:
    all_preds: List[str] = []
    all_refs: List[str] = []

    model.eval()
    for batch in tqdm(dataloader):
        input_features = batch["input_features"].to(model.device)
        labels = batch["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        references = processor.batch_decode(labels, skip_special_tokens=True)

        with torch.no_grad():
            predicted_ids = model.generate(
                input_features=input_features,
                max_new_tokens=max_new_tokens,
            )

        predictions = processor.batch_decode(predicted_ids, skip_special_tokens=True)

        for pred, ref in zip(predictions, references):
            if ref.strip():
                all_preds.append(pred)
                all_refs.append(ref)

    metrics_pct = compute_text_metrics(all_preds, all_refs)
    return {
        "wer": metrics_pct["wer"] / 100.0,
        "cer": metrics_pct["cer"] / 100.0,
    }


def show_ref_pred(
    model: WhisperForConditionalGeneration,
    processor: WhisperProcessor,
    sample: Dict[str, torch.Tensor],
    max_new_tokens: int = 225,
) -> None:
    model.eval()
    input_features = sample["input_features"].to(model.device)
    labels = sample["labels"].clone()
    labels[labels == -100] = processor.tokenizer.pad_token_id
    reference = processor.batch_decode(labels, skip_special_tokens=True)

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features=input_features,
            max_new_tokens=max_new_tokens,
        )

    prediction = processor.batch_decode(predicted_ids, skip_special_tokens=True)
    metrics_pct = compute_text_metrics(prediction, reference)

    print("Reference:", reference[0] if reference else "")
    print("Predicted:", prediction[0] if prediction else "")
    print(f"WER: {metrics_pct['wer'] / 100.0:.4f}, CER: {metrics_pct['cer'] / 100.0:.4f}")

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

eval_model = best_model.model.to(device)

metrics = evaluate_model(eval_model, processor, test_loader, max_new_tokens=225)
print(f"WER: {metrics['wer']:.4f}")
print(f"CER: {metrics['cer']:.4f}")

100%|██████████| 5542/5542 [40:13<00:00,  2.30it/s]
/var/folders/dp/y8tgtl0x59n2jb9l03_0kjww0000gn/T/ipykernel_1090/1658798283.py:82: RuntimeWarning: Neither evaluate nor jiwer is installed; using internal WER/CER fallback.
  warnings.warn(


WER: 0.5383
CER: 0.2519


## Benchmark: pretrained Whisper without fine-tuning

To put the fine-tuned WER/CER in context, we evaluate the original `openai/whisper-base` checkpoint on the same Toronto test split without any fine-tuning. This gives a zero-shot baseline for Ukrainian transcription.

In [ ]:
baseline_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-base", token=whisper_token
).to(device)

baseline_model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="Ukrainian", task="transcribe"
)
baseline_model.config.suppress_tokens = []

baseline_metrics = evaluate_model(
    baseline_model, processor, test_loader, max_new_tokens=225
)
print(f"Baseline (no fine-tuning) WER: {baseline_metrics['wer']:.4f}")
print(f"Baseline (no fine-tuning) CER: {baseline_metrics['cer']:.4f}")

In [ ]:
comparison = {
    "pretrained (no fine-tuning)": baseline_metrics,
    "fine-tuned": metrics,
}

print(f"{'Model':<32}{'WER':>10}{'CER':>10}")
print('-' * 52)
for name, m in comparison.items():
    print(f"{name:<32}{m['wer']:>10.4f}{m['cer']:>10.4f}")

wer_delta = baseline_metrics['wer'] - metrics['wer']
cer_delta = baseline_metrics['cer'] - metrics['cer']
print(f"\nWER improvement after fine-tuning: {wer_delta:+.4f}")
print(f"CER improvement after fine-tuning: {cer_delta:+.4f}")

In [ ]:
print("\nBaseline predictions on the first three test samples (no fine-tuning):\n")
for idx in range(3):
    raw = toronto_ready["test"][idx]
    batch = collator([raw])
    print(f"Sample {idx + 1}")
    show_ref_pred(baseline_model, processor, batch, max_new_tokens=225)
    print()

In [122]:
sample_1_raw = toronto_ready["test"][0]
sample_1_batch = collator([sample_1_raw])
print("\nSample 1")
show_ref_pred(eval_model, processor, sample_1_batch, max_new_tokens=225)
ipd.display(ipd.Audio(data=sample_1_raw["audio"]["array"], autoplay=False, rate=sample_1_raw["audio"]["sampling_rate"]))


Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 1
Reference: Народний депутат «Слуги народу» Роман Іванісов має погашену судимість за обвинувачення у зґвалтуванні. Так це зовсім інша справа!
Predicted: Народний депутат слугин народу Роман Іванісов має погашену судимісті за обвинуваченням у зголтування. Так це зовсім інші справи!
WER: 0.3889, CER: 0.0930


/var/folders/dp/y8tgtl0x59n2jb9l03_0kjww0000gn/T/ipykernel_1090/1658798283.py:82: RuntimeWarning: Neither evaluate nor jiwer is installed; using internal WER/CER fallback.
  warnings.warn(


In [126]:
sample_2_raw = toronto_ready["test"][1]
sample_2_batch = collator([sample_2_raw])
print("\nSample 2")
show_ref_pred(eval_model, processor, sample_2_batch, max_new_tokens=225)
ipd.display(ipd.Audio(data=sample_2_raw["audio"]["array"], autoplay=False, rate=sample_2_raw["audio"]["sampling_rate"]))


Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 2
Reference:  Де ви стоїте?...  Я кажу...  Та чорт забирай! Що сталося? Та у мене знов з таксистом дисконект.
Predicted: Деви стоїте?  Деви стоїте? І я кажу... Та чуж забирай? Ще сталося? Тому не знався таксистом «Дисконект».
WER: 0.8824, CER: 0.3542


/var/folders/dp/y8tgtl0x59n2jb9l03_0kjww0000gn/T/ipykernel_1090/1658798283.py:82: RuntimeWarning: Neither evaluate nor jiwer is installed; using internal WER/CER fallback.
  warnings.warn(


In [125]:
sample_3_raw = toronto_ready["test"][2]
sample_3_batch = collator([sample_3_raw])
print("\nSample 3")
show_ref_pred(eval_model, processor, sample_3_batch, max_new_tokens=225)
ipd.display(ipd.Audio(data=sample_3_raw["audio"]["array"], autoplay=False, rate=sample_3_raw["audio"]["sampling_rate"]))

Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sample 3
Reference: Можливо, вже прийшов час підписатись на всесвітньо відомий ютюб-канал «Телебачення Торонто»? Там є не тільки я.
Predicted: Можливо, вже прийшовся спідписатись на всесвітня відомий ютуб-канал «За лобачення Торонто». Там є не тільки я!
WER: 0.5625, CER: 0.1171


/var/folders/dp/y8tgtl0x59n2jb9l03_0kjww0000gn/T/ipykernel_1090/1658798283.py:82: RuntimeWarning: Neither evaluate nor jiwer is installed; using internal WER/CER fallback.
  warnings.warn(
